# Playbook 07 — Dashboard Demo

Walk through the new evidence-explorer pages added in Sprint 9.

Pages added to `benchmarking/dashboard.py`:
1. **Disease explorer** — pick a disease, view its single-cell signature + top candidates.
2. **Drug candidate rankings** — full filterable ranked table, tier histogram, CSV download.
3. **Evidence breakdown** — single-candidate deep dive (evidence card + reversal view + raw vector).
4. **Clinical validation** — status table for top-N candidates (known indication / trial phase / literature).

Each page pulls from artifacts written by `scripts/run_full_repurposing_pipeline.py`.

## 1. Populate the artifacts the dashboard reads

In [ ]:
import subprocess, json, sys
from pathlib import Path

# Run the full pipeline (kg+omics + validate) to produce all dashboard inputs
result = subprocess.run(
    [sys.executable, 'scripts/run_full_repurposing_pipeline.py',
     '--mode', 'kg+omics', '--validate', '--top-n', '12'],
    capture_output=True, text=True, timeout=120,
)
print('exit:', result.returncode)
print(result.stdout.splitlines()[-3:] if result.stdout else result.stderr.splitlines()[-5:])

## 2. Verify the artifacts exist

In [ ]:
for art in [
    'artifacts/predictions/top_candidates.csv',
    'artifacts/predictions/top_candidates.json',
    'artifacts/predictions/run_summary.json',
    'artifacts/predictions/final_repurposing_report.md',
]:
    p = Path(art)
    print(f'  {"✓" if p.exists() else "✗"}  {art}  ({p.stat().st_size if p.exists() else 0} bytes)')

In [ ]:
# Optional: create a synthetic disease signature so the Disease Explorer
# has signature data to render
Path('artifacts/signatures').mkdir(parents=True, exist_ok=True)
Path('artifacts/signatures/disease_signature.json').write_text(json.dumps({
    'disease': 'Disease::DOID:9352',
    'tissue': 'whole_blood',
    'cell_type': 'all_cells',
    'up_genes': [f'GENE_{i:04d}' for i in range(20)],
    'down_genes': [f'GENE_{i:04d}' for i in range(500, 520)],
    'ranked_genes': [{'gene': f'GENE_{i:04d}', 'score': 5.0 - i*0.1, 'logfc': 2.5 - i*0.1, 'p_adj': 0.001} for i in range(10)],
    'pathways': [{'pathway': 'KEGG_INSULIN_SIGNALING', 'p_value': 0.001}, {'pathway': 'KEGG_GLUCOSE_METABOLISM', 'p_value': 0.01}],
}, indent=2))
print('Synthetic signature written.')

## 3. Test the data loaders directly (same calls the dashboard makes)

In [ ]:
from benchmarking.components.data_loader import (
    load_top_candidates,
    load_run_summary,
    load_disease_signature,
    list_diseases,
    filter_by_disease,
)

df = load_top_candidates()
print(f'Loaded {len(df)} candidates with columns: {list(df.columns)[:6]}...')
print(f'Diseases: {list_diseases(df)[:5]}')

In [ ]:
summary = load_run_summary()
summary

In [ ]:
sig = load_disease_signature()
print(f'Disease signature: {sig.get("disease")} | {len(sig.get("up_genes", []))} up genes')

## 4. Launch the dashboard

From a terminal in the repo root:

```bash
streamlit run benchmarking/dashboard.py --server.port 8501
```

Then open http://localhost:8501 and use the sidebar to navigate to:
- **Disease explorer**
- **Drug candidate rankings**
- **Evidence breakdown**
- **Clinical validation**

On DGX Spark or remote hosts, use `scripts/dgx/launch_dashboard.sh` which binds to `0.0.0.0`.